# Gold Layer — Materialized Lake View (MLV)

Creates aggregated Gold tables on top of the Silver Lakehouse.  
Run this notebook **once** to create the views, then schedule it or trigger it  
after the Silver streaming notebook has populated data.

## What this notebook creates
1. **`SensorHourlyAgg`** — hourly aggregations per sensor (avg, min, max, stddev, reading count)
2. **`SensorCurrentState`** — latest reading per sensor (snapshot for status dashboards)
3. **`AlarmSummaryHourly`** — hourly alarm counts per unit and alarm type
4. **`CompressorKpiHourly`** — hourly KPI rollup per compressor (power, discharge temp, compression ratio)

## Approach
Uses Spark batch reads from Silver Delta tables and writes to Gold as Delta tables.  
For live updates, schedule this notebook every 15 minutes via a Fabric Data Pipeline,  
or create a **Materialized Lake View** item directly in Fabric.

## Prerequisites
- `01-silver-streaming.ipynb` has been running long enough to populate Silver tables
- Gold Lakehouse created (empty, or same as Silver — set below)

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
SILVER_LAKEHOUSE = "silver_lakehouse"
GOLD_LAKEHOUSE   = "gold_lakehouse"    # Can be same as silver if you want one lakehouse

LOOKBACK_HOURS   = 72   # How many hours of Silver data to aggregate into Gold on each run

In [ ]:
from notebookutils import mssparkutils
from pyspark.sql import functions as F
from delta.tables import DeltaTable

silver_path = mssparkutils.lakehouse.get(SILVER_LAKEHOUSE).properties.abfsPath
gold_path   = mssparkutils.lakehouse.get(GOLD_LAKEHOUSE).properties.abfsPath

silver_tables = f"{silver_path}/Tables"
gold_tables   = f"{gold_path}/Tables"

print(f"Silver : {silver_tables}")
print(f"Gold   : {gold_tables}")

In [ ]:
# ── Load Silver SensorReadings (bounded window for incremental runs) ───────────
from datetime import datetime, timedelta, timezone

cutoff = datetime.now(timezone.utc) - timedelta(hours=LOOKBACK_HOURS)

df_silver = (
    spark.read.format("delta")
    .load(f"{silver_tables}/SensorReadings")
    .filter(F.col("ts") >= F.lit(cutoff))
    .cache()
)
print(f"Silver readings loaded (last {LOOKBACK_HOURS}h): {df_silver.count():,} rows")

In [ ]:
# ── Gold Table 1: SensorHourlyAgg ─────────────────────────────────────────────
# One row per sensor per hour — avg, min, max, std, count, out-of-range count
df_hourly = (
    df_silver
    .withColumn("hour", F.date_trunc("hour", F.col("ts")))
    .groupBy(
        "hour", "sensor_id", "tag_id", "tag_name",
        "parameter_type", "unit_of_measure",
        "equipment_id", "equipment_name", "equipment_type"
    )
    .agg(
        F.round(F.avg("value"),    3).alias("avg_value"),
        F.round(F.min("value"),    3).alias("min_value"),
        F.round(F.max("value"),    3).alias("max_value"),
        F.round(F.stddev("value"), 3).alias("stddev_value"),
        F.count("*").alias("reading_count"),
        F.sum(F.col("is_out_of_range").cast("integer")).alias("out_of_range_count"),
        F.sum((F.col("alarm_status") != "NORMAL").cast("integer")).alias("alarm_event_count")
    )
)

# Write using merge (upsert) to avoid duplicates on re-runs
gold_hourly_path = f"{gold_tables}/SensorHourlyAgg"

try:
    dt = DeltaTable.forPath(spark, gold_hourly_path)
    (
        dt.alias("target")
        .merge(
            df_hourly.alias("source"),
            "target.hour = source.hour AND target.sensor_id = source.sensor_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"SensorHourlyAgg merged successfully.")
except Exception:
    df_hourly.write.format("delta").mode("overwrite").save(gold_hourly_path)
    print(f"SensorHourlyAgg created (first run).")

In [ ]:
# ── Gold Table 2: SensorCurrentState ─────────────────────────────────────────
# Latest reading per sensor — used by status tiles and DirectLake semantic model
from pyspark.sql.window import Window

w = Window.partitionBy("sensor_id").orderBy(F.desc("ts"))

df_current = (
    df_silver
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "reading_id")
    .withColumn("snapshot_ts", F.current_timestamp())
)

(
    df_current.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{gold_tables}/SensorCurrentState")
)
print(f"SensorCurrentState written: {df_current.count()} rows (one per active sensor)")

In [ ]:
# ── Gold Table 3: CompressorKpiHourly ────────────────────────────────────────
# Hourly KPIs per compressor: avg power, avg discharge temp, compression ratio
df_comp = (
    df_silver
    .filter(F.col("equipment_type") == "Compressor")
    .withColumn("hour", F.date_trunc("hour", F.col("ts")))
)

# Pivot by parameter_type and aggregate
df_kpi = (
    df_comp
    .groupBy("hour", "equipment_id", "equipment_name")
    .agg(
        F.round(F.avg(F.when(F.col("parameter_type") == "Power",       F.col("value"))), 2).alias("avg_power_kw"),
        F.round(F.avg(F.when(F.col("parameter_type") == "RPM",         F.col("value"))), 1).alias("avg_rpm"),
        F.round(F.avg(F.when(
            (F.col("parameter_type") == "Temperature") & F.col("tag_id").contains("TT-002"),
            F.col("value")
        )), 2).alias("avg_discharge_temp_c"),
        F.round(F.avg(F.when(
            (F.col("parameter_type") == "Pressure") & F.col("tag_id").endswith("PT-002"),
            F.col("value")
        )), 2).alias("avg_discharge_pressure_bar"),
        F.round(F.avg(F.when(
            (F.col("parameter_type") == "Pressure") & F.col("tag_id").endswith("PT-001"),
            F.col("value")
        )), 2).alias("avg_suction_pressure_bar")
    )
    .withColumn(
        "compression_ratio",
        F.round(F.col("avg_discharge_pressure_bar") / F.col("avg_suction_pressure_bar"), 3)
    )
)

gold_kpi_path = f"{gold_tables}/CompressorKpiHourly"
try:
    dt = DeltaTable.forPath(spark, gold_kpi_path)
    (
        dt.alias("t")
        .merge(df_kpi.alias("s"),
               "t.hour = s.hour AND t.equipment_id = s.equipment_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("CompressorKpiHourly merged.")
except Exception:
    df_kpi.write.format("delta").mode("overwrite").save(gold_kpi_path)
    print("CompressorKpiHourly created (first run).")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("\n=== Gold Layer Summary ===")
for tbl in ["SensorHourlyAgg", "SensorCurrentState", "CompressorKpiHourly"]:
    path = f"{gold_tables}/{tbl}"
    try:
        n = spark.read.format("delta").load(path).count()
        print(f"  {tbl:<30} {n:>10,} rows")
    except Exception as e:
        print(f"  {tbl:<30} ERROR: {e}")
print("\nNext step: Create a DirectLake Semantic Model on these Gold tables in Fabric.")